In [1]:
import sys
import re
import numpy as np
import pandas as pd
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests
from neutrality_test import *

In [2]:
raw = pd.read_csv("Table-S7-sgRNA-Raw-Counts.csv")
sample_names = raw.iloc[0]

def normalize_sgrna_id(x) -> str: #Convert IDs like 'sgRNA1', 'sgRNA001', 1 -> 'sgRNA0001'. If no digits are found, returns the original string."""
    s = str(x).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        return s
    num = m.group(1)
    return f"sgRNA{num.zfill(4)}"

# --- normalized sgRNA IDs (length K) ---
sgRNAs_raw = raw.iloc[1:, 0].astype(str).to_numpy()
sgRNAs = np.array([normalize_sgrna_id(v) for v in sgRNAs_raw], dtype=object)

In [3]:
def counts_for_cols(cols):
    """Return a (K x len(cols)) DataFrame of integer counts with index=normalized sgRNAs."""
    mat = raw.loc[1:, cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    mat.index = sgRNAs
    return mat.astype(np.int64)

# ---- donor = Pre1 (from 'preinfection samples for 24 hpi...') ----
j0 = raw.columns.get_loc("preinfection samples for 24 hpi of murine pneumonia model")
pre_cols = list(raw.columns[j0 : j0 + 2])  # (Pre1, Pre2)
donor_col = next(c for c in pre_cols if str(sample_names[c]) == "Pre1")
donor_counts = counts_for_cols([donor_col]).iloc[:, 0]  # pd.Series, index=sgRNA####

# ---- recipients = 24 hpi mice WITH dox (Mice_noDox_*) ----
i0 = raw.columns.get_loc("24 hpi of murine pneumonia model")
i1 = raw.columns.get_loc("preinfection samples for 48 hpi of murine pneumonia model")
cols_24hpi = list(raw.columns[i0:i1])

nodox_cols = [c for c in cols_24hpi if isinstance(sample_names[c], str) and sample_names[c].startswith("Mice_noDox_")]

recipient_counts_df = counts_for_cols(nodox_cols).T
recipient_counts_df.index = [sample_names[c] for c in nodox_cols]   # mice IDs # recipient_counts_df.columns are already normalized sgRNA####

In [4]:
nb = pd.read_csv("Leonard_Nb_24hpi.csv")
nb_dox = nb[(nb["hpi"] == 24) & (nb["condition"] == "noDox")].copy()
nb_dox["sample"] = nb_dox["mouse"].astype(int).map(lambda m: f"Mice_noDox_{m}")
Nb_by_mouse = dict(zip(nb_dox["sample"], nb_dox["lung"]))

missing = [m for m in recipient_counts_df.index if m not in Nb_by_mouse]
if missing:
    raise ValueError(f"Missing Nb entries for these recipients: {missing}")

Nb_vec = np.array([Nb_by_mouse[m] for m in recipient_counts_df.index], dtype=float)

In [6]:
#run neutrality test
res = neutrality_lrt(donor_counts = donor_counts, recipient_counts = recipient_counts_df, bottleneck = Nb_vec,)

res["feature"] = res["feature"].map(normalize_sgrna_id)
sig = res[res["reject_FDR"]].copy()
sig.to_csv("neutrality_test_24hpi_noDox.csv", index=False)